# DepthWizard v4 — HF download test (Modal Notebook, **CPU**)

Does the gated encoder download, and does it build? Nothing else. No repo
clone, no data Volume, no training code — so the only things that can fail here
are the things this notebook is testing.

`02_train.ipynb` cell 4 is where this normally happens, on an H100 at
**~$4.90/h**. This is the same sequence on a CPU box at a few cents an hour.

## It is not only a test — it warms the cache

Cell 3 points `HF_HOME` at `/mnt/depthwizard-results/hf`, which is exactly what
`02_train.ipynb` cell 3 does. That Volume is persistent, so the ~1.2 GiB this
notebook pulls is **the copy the H100 run will find already there**. Downloading
it here rather than there is the cheaper way round, and it means cell 4 on the
training kernel is a cache hit instead of a download.

Attach the Volume, or the download lands on container disk and dies with the
kernel — still a valid test, just a wasted 1.2 GiB.

## Before you run a single cell

| Sidebar tab | Set to |
|---|---|
| Compute profile | **2–4 CPU, 8–16 GiB, no GPU** — this is a download, not a workload |
| Filesystem | Volume `depthwizard-results` → `/mnt/depthwizard-results` |
| Secrets (or cell 1) | `dw-hf` (`HF_TOKEN`) |

## What each cell answers

| cell | question |
|---|---|
| 4 | Is the token reaching HF at all, and whose is it? |
| 5 | Does *this* token read gated content — proved against an anonymous control? |
| 6 | Does the full 1.2 GiB snapshot actually land? |
| 7 | Does `transformers` build the real DINOv3 from it? |

Cell 5 is the one that earns its place. A 401 on a gated repo looks identical
whether the token is absent, invalid, or merely unprivileged — the anonymous
control is what tells those apart, and it is what settled this last time.

In [ ]:
# 1. Credentials — paste the token here.
#
# LEAVE THIS EMPTY IN THE COMMITTED COPY. Fill it in on YOUR copy inside Modal.
# Blank falls through to a Modal Secret named HF_TOKEN (the dw-hf secret), so
# attaching it in the sidebar works too — but note the precedence: a value typed
# here WINS over the Secret. That is deliberate. It is how you test a new token
# without touching the Secret, and it is how you rule the Secret out when the
# two have drifted apart.
HF_TOKEN = ""                 # huggingface.co/settings/tokens

import os

if HF_TOKEN:
    # .strip(): a trailing newline is what you get pasting out of a terminal, a
    # file or a Secret, and HF sends it verbatim then answers 401 "Invalid
    # username or password" — which reads as a revoked token, not whitespace.
    os.environ["HF_TOKEN"] = HF_TOKEN.strip()

_t = os.environ.get("HF_TOKEN", "")
assert _t, ("HF_TOKEN is not set — paste it into this cell, or attach the "
            "dw-hf Secret in the sidebar")
# Never print the token. A fingerprint is enough to compare two copies of it,
# and this notebook's whole job is telling two copies apart.
print(f"[dw] HF_TOKEN present: len={len(_t)} prefix={_t[:3]!r} last4={_t[-4:]!r} "
      f"source={'this cell' if HF_TOKEN else 'Modal Secret / environment'}")
if len(_t) != 37 or not _t.startswith("hf_"):
    print(f"[dw] !! expected 37 chars starting 'hf_' — this one is {len(_t)}. "
          f"Truncated paste, or not an HF token.")

In [ ]:
# 2. Deps. Only what the download path needs — no requirements.txt, no repo.
#
# torch AND torchvision in ONE resolve off the pytorch index. transformers
# imports torchvision, and a pair built against different torches raises
# "operator torchvision::nms does not exist" (or, on some images, "partially
# initialized module 'torchvision' has no attribute 'extension'") — which takes
# AutoModel with it and would look exactly like a download failure here.
# torchvision stays unpinned so the torch pin picks its matching build.
%uv pip install torch==2.10.0 torchvision --index-url https://download.pytorch.org/whl/cpu
%uv pip install "transformers>=4.56.0" "huggingface_hub>=0.34.0" safetensors
# Shell form, if the %uv magic rejects a flag:
#   !uv pip install --system torch==2.10.0 torchvision --index-url https://download.pytorch.org/whl/cpu
#   !uv pip install --system "transformers>=4.56.0" "huggingface_hub>=0.34.0" safetensors

In [ ]:
# 3. Where the download goes. This is the cache 02_train.ipynb will read.
from pathlib import Path

ENCODER = "facebook/dinov3-vitl16-pretrain-sat493m"
RESULTS = "/mnt/depthwizard-results"

# Identical to 02_train.ipynb cell 3. If it differs, the H100 re-downloads.
if Path(RESULTS).is_dir():
    os.environ["HF_HOME"] = f"{RESULTS}/hf"
    _where = "the PERSISTENT Volume — the H100 run will find this cached"
else:
    _where = ("container disk — the Volume is NOT attached, so this download "
              "dies with the kernel and the H100 pays for it again")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "0"     # this is the one time you want them

HF_HOME = os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface"))
Path(HF_HOME).mkdir(parents=True, exist_ok=True)
print(f"[dw] HF_HOME = {HF_HOME}\n     {_where}")

# torchvision first: a broken one makes cell 7 fail in a way that looks like a
# download problem, and it is not.
import torch
try:
    import torchvision
    from torchvision.ops import nms
    nms(torch.zeros(0, 4), torch.zeros(0), 0.5)
    print(f"[dw] torch {torch.__version__} + torchvision {torchvision.__version__} — "
          f"pair loads")
except ImportError:
    print("[dw] no torchvision — fine, transformers falls back to PIL")
except Exception as e:                    # noqa: BLE001
    raise RuntimeError(
        f"[dw] torchvision is ABI-broken against torch {torch.__version__}: {e}\n"
        f"     Re-run cell 2; if it persists add --reinstall to that line.")

VERDICT = {}

In [ ]:
# 4. Is the token reaching HF, and whose is it?
#
# whoami-v2 is the cheapest possible question: it needs no repo access at all,
# so a 401 HERE means the credential itself is being rejected and nothing about
# gating is in play yet.
from huggingface_hub import HfApi

api = HfApi(token=os.environ["HF_TOKEN"])
try:
    me = api.whoami()
    a = (me.get("auth") or {}).get("accessToken") or {}
    print(f"[4] account : {me.get('name')}")
    print(f"[4] token    : {a.get('displayName')}  role={a.get('role')}")
    fg = a.get("fineGrained")
    if fg:
        # Worth reading rather than skimming. A fine-grained token is an
        # allow-list; `repo.content.read` scoped to your OWN user does not grant
        # anything on facebook/*. What covers a gated repo you do not own is a
        # GLOBAL read permission.
        print("[4] fine-grained scopes:")
        for g in fg.get("global") or []:
            print(f"      global: {g}")
        for s in fg.get("scoped") or []:
            ent = s.get("entity") or {}
            print(f"      scoped: {ent.get('type')} {ent.get('name')} -> "
                  f"{s.get('permissions')}")
    else:
        print("[4] classic token — inherits every permission the account has")
    VERDICT["4 identity"] = []
except Exception as e:                    # noqa: BLE001
    VERDICT["4 identity"] = [f"whoami failed: {type(e).__name__}: {str(e)[:200]}"]
    print(f"[4] !! {VERDICT['4 identity'][0]}\n"
          f"     The credential is rejected outright. Nothing about gating or "
          f"licences is involved yet — the value in HF_TOKEN is wrong, "
          f"truncated, or revoked. Compare the fingerprint from cell 1 against "
          f"the token you meant to use.")

In [ ]:
# 5. Does THIS token read gated content? Proved against an anonymous control.
#
# Why the control: a gated repo serves README.md and LICENSE.md publicly, so you
# can read the terms before accepting, and 401s everything else. A request with
# a broken token therefore looks EXACTLY like a request with no token —
# README 200, config.json 401. Without the anonymous column that pattern reads
# as "the token lacks permission" when it can equally mean "the token never
# arrived". The columns tell them apart:
#
#   token 200 / anon 401  -> the token works. Gating is not your problem.
#   token 401 / anon 401  -> the token is not granting access (absent, invalid,
#                            unprivileged, or the licence is unaccepted).
#   token 401 / anon 200  -> you are looking at a public file; it proves nothing.
import requests

BASE = f"https://huggingface.co/{ENCODER}/resolve/main"
HDR = {"Authorization": f"Bearer {os.environ['HF_TOKEN']}"}
problems = []

print(f"{'file':<28}{'with token':>12}{'anonymous':>12}")
for f in ("README.md", "config.json", "preprocessor_config.json",
          "model.safetensors"):
    # HEAD, not GET: model.safetensors is 1.2 GiB and this cell is a permission
    # check, not the download. allow_redirects follows to the CDN, which is
    # where the real answer comes from.
    try:
        a = requests.head(f"{BASE}/{f}", headers=HDR, allow_redirects=True, timeout=30)
        b = requests.head(f"{BASE}/{f}", allow_redirects=True, timeout=30)
        print(f"{f:<28}{a.status_code:>12}{b.status_code:>12}")
        if f != "README.md" and a.status_code != 200:
            problems.append(f"{f}: {a.status_code} with the token "
                            f"({b.status_code} anonymous)")
    except Exception as e:                # noqa: BLE001
        problems.append(f"{f}: request failed: {type(e).__name__}: {e}")
        print(f"{f:<28}{'ERROR':>12}{'ERROR':>12}")

try:
    info = api.model_info(ENCODER)
    print(f"\n[5] {info.id}  gated={info.gated}  private={info.private}")
except Exception as e:                    # noqa: BLE001
    problems.append(f"model_info: {type(e).__name__}: {str(e)[:160]}")

if problems:
    print("\n[5] !! " + "\n       ".join(problems))
    print("\n     If the anonymous column is ALSO 401 on those files, the token "
          "\n     is not granting access. In order of likelihood:"
          "\n       1. HF_TOKEN here is not the token you think (cell 1's"
          "\n          fingerprint vs the one you meant)."
          "\n       2. The account has not accepted the licence:"
          f"\n          https://huggingface.co/{ENCODER}"
          "\n       3. A fine-grained token without a GLOBAL gated-read"
          "\n          permission (cell 4 printed its scopes). A classic `read`"
          "\n          token sidesteps this.")
VERDICT["5 gated access"] = problems

In [ ]:
# 6. The real download. ~1.2 GiB; minutes on a CPU box, and free to re-run —
#    snapshot_download skips what is already complete on disk.
import time

from huggingface_hub import snapshot_download

t0 = time.time()
try:
    path = snapshot_download(ENCODER, token=os.environ["HF_TOKEN"])
    gib = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 2**30
    print(f"\n[6] snapshot -> {path}\n    {gib:.2f} GiB in {(time.time() - t0) / 60:.1f} min")
    # The weights are the point. A snapshot of just the config and README is
    # what a gated repo hands an unauthorised client, and it would otherwise
    # look like success.
    weights = [f.name for f in Path(path).rglob("*")
               if f.suffix in (".safetensors", ".bin")]
    print(f"    weight files: {weights or 'NONE'}")
    VERDICT["6 download"] = ([] if weights else
                             ["the snapshot carries no .safetensors/.bin — "
                              "config and README only, which is what an "
                              "unauthorised client gets"])
except Exception as e:                    # noqa: BLE001
    VERDICT["6 download"] = [f"{type(e).__name__}: {str(e)[:300]}"]
    print(f"\n[6] !! {VERDICT['6 download'][0]}")

In [ ]:
# 7. Does it BUILD? A downloaded snapshot is not a working encoder.
#
# This is the check 02_train.ipynb cell 4 makes before spending anything on
# training, reproduced here. It is also the one a green test suite cannot make:
# every test but tests/test_real_dinov3.py runs against a stub encoder.
try:
    from transformers import AutoImageProcessor
    proc = AutoImageProcessor.from_pretrained(ENCODER, token=os.environ["HF_TOKEN"])
    print(f"[7] processor: mean={getattr(proc, 'image_mean', None)} "
          f"std={getattr(proc, 'image_std', None)}")
    print("    ^ 02_train reads these as the preprocessing contract; without "
          "them preprocess.py falls back to constants")
except Exception as e:                    # noqa: BLE001
    print(f"[7] processor unavailable ({type(e).__name__}: {str(e)[:120]}) — "
          f"the constant fallback applies, not fatal")

problems = []
try:
    from transformers import AutoModel
    enc = AutoModel.from_pretrained(ENCODER, token=os.environ["HF_TOKEN"])
    c = enc.config
    print(f"[7] built {type(enc).__name__}  hidden={c.hidden_size} "
          f"patch={c.patch_size} blocks={c.num_hidden_layers}")
    want = ("DINOv3ViTModel", 1024, 16, 24)
    got = (type(enc).__name__, c.hidden_size, c.patch_size, c.num_hidden_layers)
    if got != want:
        problems.append(f"expected {want}, got {got} — this is not the ViT-L/16 "
                        f"SAT-493M encoder the config names")
    del enc
except Exception as e:                    # noqa: BLE001
    problems.append(f"{type(e).__name__}: {str(e)[:300]}")
    print(f"[7] !! {problems[-1]}")
VERDICT["7 build"] = problems

In [ ]:
# 8. The verdict.
for k in ("4 identity", "5 gated access", "6 download", "7 build"):
    state = ("not run" if k not in VERDICT
             else "ok" if not VERDICT[k] else f"{len(VERDICT[k])} problem(s)")
    print(f"  {k:<16} {state}")

bad = {k: v for k, v in VERDICT.items() if v}
missing = [k for k in ("4 identity", "5 gated access", "6 download", "7 build")
           if k not in VERDICT]
print()
if bad:
    for k, v in bad.items():
        for p in v:
            print(f"!! [{k}] {p}")
    print("\nDo not start 02_train.ipynb — cell 4 there makes these same checks "
          "on a billed H100.")
elif missing:
    print(f"No problems, but {missing} did not run — that is not a pass.")
else:
    print("=" * 66)
    print("ALL CLEAR — the encoder downloads and builds. 02_train cell 4 will pass.")
    print("=" * 66)
    if Path(RESULTS).is_dir():
        _hf = Path(os.environ["HF_HOME"])
        _g = sum(f.stat().st_size for f in _hf.rglob("*") if f.is_file()) / 2**30
        print(f"Cached on the Volume: {_hf}  ({_g:.2f} GiB)")
        print("The H100 kernel will reuse this — cell 4 there becomes a cache hit.")
    else:
        print("!! depthwizard-results was NOT attached, so this download is "
              "about to die with the kernel.\n"
              "   Attach it and re-run cell 6 if you want the H100 to reuse it.")

# This kernel is cheap, not free. Sidebar -> Stop kernel.